In [7]:
import pandas as pd
import numpy as np
import psycopg
import hashlib

from decimal import Decimal
from pathlib import Path

from getpass import getpass
import psycopg

pg_password = getpass("PostgreSQL password: ")

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="sales_etl",
    user="postgres",
    password=pg_password
)

print("conn deone")


PostgreSQL password:  ········


conn deone


In [8]:
def start_audit(batch_id, source_file):
    query = """
        INSERT INTO audit.etl_batch_log (
            batch_id,
            source_file,
            status,
            started_at,
            completed_at,
            error_message
        )
        VALUES (%s, %s, %s, CURRENT_TIMESTAMP, NULL, NULL)
        ON CONFLICT (batch_id)
        DO UPDATE SET
            source_file = EXCLUDED.source_file,
            status = 'STARTED',
            started_at = CURRENT_TIMESTAMP,
            completed_at = NULL,
            error_message = NULL;
    """

    with conn.cursor() as cur:
        cur.execute(
            query,
            (batch_id, source_file, "STARTED")
        )

    conn.commit()
    print(f"Batch {batch_id} started")


def complete_audit(batch_id):
    query = """
        UPDATE audit.etl_batch_log
        SET status = 'SUCCESS',
            completed_at = CURRENT_TIMESTAMP,
            error_message = NULL
        WHERE batch_id = %s;
    """

    with conn.cursor() as cur:
        cur.execute(query, (batch_id,))

    conn.commit()
    print(f"Batch {batch_id} completed successfully")


def fail_audit(batch_id, error_message):
    query = """
        UPDATE audit.etl_batch_log
        SET status = 'FAILED',
            completed_at = CURRENT_TIMESTAMP,
            error_message = %s
        WHERE batch_id = %s;
    """

    with conn.cursor() as cur:
        cur.execute(
            query,
            (str(error_message)[:4000], batch_id)
        )

    conn.commit()
    print(f"Batch {batch_id} failed")


In [9]:
batch_id = "audit_test_001"
source_file = "sales_2026_10_28.csv"

start_audit(batch_id, source_file)

try:
    # Simulate a successful ETL operation
    print("ETL processing completed")

    complete_audit(batch_id)

except Exception as e:
    conn.rollback()
    fail_audit(batch_id, e)
    raise

Batch audit_test_001 started
ETL processing completed
Batch audit_test_001 completed successfully
